# STEP 3 — 전처리

**순서가 중요합니다.** 중복 제거 → 그룹 분할 → 크롭 순서로 갑니다.

```
JSON 파싱 ──▶ 매니페스트(표 한 장)
                  │
                  ▼
            중복 제거 (phash)        ← 라벨 충돌하는 사진 제거
                  │
                  ▼
            개체 단위 분할            ← ★ 정확도 신뢰성의 핵심
                  │
                  ▼
            ROI 크롭                 ← 병변 주변만 잘라내기
```

**왜 이 순서인가?**
중복을 먼저 안 없애면, 같은 사진이 train 과 val 에 나뉘어 들어가서
분할을 아무리 잘해도 새어버립니다.

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 clone → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"          # 작업 브랜치를 쓰려면 여기만 바꾸세요
DIR    = "deeplearning_test"

if os.path.basename(os.getcwd()) != DIR and not os.path.exists("src"):
    if os.path.exists(DIR):
        subprocess.run(["git", "-C", DIR, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO], check=True)
    os.chdir(DIR)
sys.path.insert(0, os.getcwd())
print("작업 디렉터리:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "imagehash", "pyarrow", "grad-cam"], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
if not os.path.exists("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

## 1. 매니페스트 만들기

모든 라벨 JSON 을 읽어 **표 한 장**으로 만듭니다.
이후 모든 단계는 이 표만 보고 동작합니다 — 폴더를 다시 뒤지지 않으니 재현성이 생깁니다.

In [ ]:
from src import labels, scan

rep = scan.ScanReport.load()      # STEP 2의 스캔 결과 재사용
df = labels.build(report=rep)
df.head(3)

### 확인할 것

- `bbox 있음` / `polygon 있음` 비율이 0% 면 좌표 추출이 실패한 것입니다
- `개체당 평균 장수`가 1.5 미만이면 개체ID 추출이 실패한 것입니다

둘 중 하나라도 문제면 아래에서 수동으로 잡아주세요.

In [ ]:
# 수동 지정 예시 — 자동 추정이 틀렸을 때
# df = labels.build(report=rep, animal_token_index=2)

print("bbox 있음    :", f"{df['bbox'].notna().mean():.1%}")
print("polygon 있음 :", f"{df['polygon'].notna().mean():.1%}")
print("개체 수      :", f"{df['animal_id'].nunique():,}")
print("개체당 평균  :", f"{len(df)/max(df['animal_id'].nunique(),1):.1f}장")

## 2. 좌표가 맞는지 눈으로 확인 ★

**절대 건너뛰지 마세요.** 좌표계 실수(x/y 뒤바뀜, 정규화 좌표를 픽셀로 착각,
좌상단 기준 vs 중심 기준)는 **그림을 봐야만** 발견됩니다.

숫자로는 아무 이상이 없어 보이는데 크롭이 전부 엉뚱한 곳을 자르고 있으면,
학습은 잘 돌고 정확도만 안 나와서 원인을 못 찾습니다.

In [ ]:
from src import crop
crop.preview_with_box(df, n=4)

## 3. 중복 제거

`phash`(perceptual hash)는 파일이 달라도 **보기에 같은** 이미지를 잡아냅니다.
리사이즈, 약한 JPEG 재압축, 미세한 밝기 차이를 견딥니다.

두 가지를 처리합니다:
1. **라벨이 충돌하는 중복** → 전부 제거 (어느 쪽이 맞는지 알 수 없는 오염 데이터)
2. **같은 라벨의 중복** → 대표 1장만 남김

In [ ]:
from src import dedup

cfg = CFG()
df, dup_info = dedup.run(df, cfg)
dup_info

## 4. 개체 단위 분할 ★★★

이 프로젝트에서 가장 중요한 셀입니다.

**왜?** 이 데이터는 강아지 1마리당 수십 장을 찍었습니다.
같은 강아지 사진이 train 과 val 에 나뉘어 들어가면 모델은 병변이 아니라
**"이 강아지"를 외워서** 맞힙니다. 검증 정확도 95%가 나와도 처음 보는
강아지에서는 60%밖에 안 나옵니다.

코드는 멀쩡히 돌고 **숫자만 거짓말을 하기 때문에** 배포하고 나서야 압니다.

여기서는 `개체ID ∪ 중복클러스터`를 하나의 그룹으로 묶어서 나눕니다.

In [ ]:
from src import split

df = split.assign(df, cfg)
split.verify(df, fold=0)          # 누수가 있으면 여기서 에러가 납니다

In [ ]:
# 만약 이미지 단위로 대충 나눴다면 얼마나 샜을지 — 참고용
split.compare_with_random(df, cfg)

## 5. ROI 크롭

병변이 이미지의 5% 미만이라면 전체 이미지를 넣는 건 낭비를 넘어 **해롭습니다.**
모델이 병변 대신 배경(진료대, 손, 목줄, 조명)을 학습하기 때문입니다.

여기서는 **세 가지 버전**을 만들어 STEP 4에서 비교합니다:

| tag | 의미 |
|---|---|
| `m1.5` | 병변 박스 1.5배 — 병변 위주, 주변 맥락 조금 |
| `m2.5` | 병변 박스 2.5배 — 주변 피부 맥락 더 포함 |
| `full` | 크롭 없이 중앙 정사각 — 대조군 |

어느 게 좋을지는 미리 알 수 없습니다. **실험으로 정합니다.**

In [ ]:
df_m15 = crop.run(df, cfg, margin=1.5)
labels.save(df_m15, "manifest_m1.5.parquet")

In [ ]:
df_m25 = crop.run(df, cfg, margin=2.5)
labels.save(df_m25, "manifest_m2.5.parquet")

In [ ]:
df_full = crop.run(df, cfg, margin=0)     # margin=0 → 중앙 정사각 크롭
labels.save(df_full, "manifest_full.parquet")

## 6. 크롭 결과 확인 ★

병변이 화면 안에 잘 들어와 있는지 봅니다.

In [ ]:
crop.preview(df_m15, n=8)

## 7. 최종 점검

In [ ]:
import pandas as pd

tr, va = split.get_fold(df_m15, 0)
ho = split.get_holdout(df_m15)

print(f"train {len(tr):,} / val {len(va):,} / holdout {len(ho):,}\n")
print(pd.crosstab(df_m15["label"], df_m15["fold"]).to_string())
print()
dedup.sanity_check_split(tr, va)

### Drive 백업 (Colab)

크롭본과 매니페스트만 백업합니다. 원본은 너무 큽니다.

In [ ]:
# DRIVE = env.mount_drive()
# import shutil
# shutil.copytree(env.work_root()/"manifests", DRIVE/"dogskin/manifests", dirs_exist_ok=True)
# shutil.make_archive(str(DRIVE/"dogskin/crops_m1.5"), "zip", env.work_root()/"crops/m1.5")

---
## ✅ 다음 단계

`03_학습_베이스라인.ipynb`

📖 함께 읽기 (전처리를 왜 이렇게 했는지 이해하려면):
- [`docs/cautions/02_데이터_누수_가장_치명적인_함정.md`](../docs/cautions/02_데이터_누수_가장_치명적인_함정.md)
- [`docs/basics/06_과적합_정규화_데이터증강.md`](../docs/basics/06_과적합_정규화_데이터증강.md)